In [ ]:
print('===== NATIVE LAYERNORM FWD BITEXACT TEST (PURE TKV, NO C INJECTION) =====')
print('===== NATIVE EMBED BWD BITEXACT TEST (C helper injected) =====')
print('===== NATIVE SOFTMAX CB V2 - CTYPES FIXED =====')
print('===== NATIVE SOFTMAX CAUSAL BWD BITEXACT TEST =====')
print('===== NATIVE SOFTMAX CAUSAL FWD BITEXACT TEST =====')
print('===== NATIVE MERGEHBHEAD BITEXACT TEST =====')
# 1/4 - clone main + prototype (BiasAddRows .tkv)
import subprocess, os
print('===== NATIVE BIASADD BITEXACT TEST =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so (LayernormFwd f64 fallback) + .tkv C .so. NO C injection.
# QUAN TRONG: build CA HAI bang CUNG flags de FMA contraction khong lam lech bit.
import subprocess, os
FLAGS='-O2 -mavx2 -mfma'
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc '+FLAGS+' -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])

ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/layernorm_fwd_cpu.cpu.c'; so=ex+'/layernorm_fwd_cpu.so'
txt=open(cc).read()
print('C INJECTED =', 'tkv_lnfwd' in txt, '(expect False = pure TKV)')
print('sqrt(double) present =', 'sqrt(d_var' in txt)
p2=subprocess.run('gcc '+FLAGS+' -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

# xac nhan ATen bridge vang mat -> native.LayernormFwd dung f64 fallback (khong phay ATen)
import ctypes, numpy as np
probe=ctypes.CDLL(root+'/tkv_simd_native.so').TkvTorchLayerNormFwd
PP=ctypes.POINTER(ctypes.c_float)
probe.argtypes=[PP,PP,PP,PP,ctypes.c_longlong,ctypes.c_longlong,ctypes.c_double]
probe.restype=ctypes.c_int
z=np.zeros(8,np.float32)
rc=probe(z.ctypes.data_as(PP),z.ctypes.data_as(PP),z.ctypes.data_as(PP),z.ctypes.data_as(PP),2,4,1e-6)
print('TkvTorchLayerNormFwd rc=',rc,'(!=0 => native used f64 fallback)')

In [ ]:
# 3/4 - bitwise compare LayernormFwd (.tkv pure vs native f64 fallback)
import ctypes, numpy as np
PP=ctypes.POINTER(ctypes.c_float); II=ctypes.c_int
args=[PP,II,PP,II,PP,II,PP,II,II,II]
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so').LayernormFwd
native.argtypes=args; native.restype=II
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/layernorm_fwd_cpu.so').LayernormFwd
tkv.argtypes=args; tkv.restype=II
rng=np.random.default_rng(11)
for n,h in [(4,64),(16,256),(8,768),(7,33)]:
    x=rng.standard_normal(n*h).astype(np.float32)
    g=rng.standard_normal(h).astype(np.float32)
    b=rng.standard_normal(h).astype(np.float32)
    a=np.zeros(n*h,np.float32); c=np.zeros(n*h,np.float32)
    native(x.ctypes.data_as(PP),x.size,g.ctypes.data_as(PP),g.size,b.ctypes.data_as(PP),b.size,a.ctypes.data_as(PP),a.size,n,h)
    tkv (x.ctypes.data_as(PP),x.size,g.ctypes.data_as(PP),g.size,b.ctypes.data_as(PP),b.size,c.ctypes.data_as(PP),c.size,n,h)
    d=np.count_nonzero(a.view(np.uint32)!=c.view(np.uint32))
    print('n=%d h=%d bitwise=%s diff=%d/%d maxabs=%.3g'%(n,h,d==0,d,a.size,np.abs(a-c).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> LayernormFwd pure .tkv reproduces native f64 fallback (NO C injection).')